[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/indianspeedster/TorchCode/blob/master/templates/24_rope.ipynb)

# 🔴 Hard: Rotary Position Embedding (RoPE)

Implement **RoPE** — the position encoding used in LLaMA, GPT-NeoX, and most modern LLMs.

### Signature
```python
def apply_rope(q: Tensor, k: Tensor) -> tuple[Tensor, Tensor]:
    # q, k: (B, S, D) where D is even
    # Returns rotated (q, k) with same shape
```

### Key Idea
Split each vector into consecutive pairs. Rotate each pair by `θ = pos / 10000^(2i/D)`:
```
[x_0, x_1] → [x_0*cosθ - x_1*sinθ, x_0*sinθ + x_1*cosθ]
```
This makes `dot(q_rot[i], k_rot[j])` depend only on `i - j` (relative position).

In [1]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [2]:
import torch
import math

In [9]:
# ✏️ YOUR IMPLEMENTATION HERE

def apply_rope(q, k):
    # 1. Compute position angles
    B,N,Dh = q.shape
    idx = torch.arange(0, Dh, 2).float()
    inv_freq = 1000**(-idx/Dh)
    positions = torch.arange(N).float()
    angles = positions[:,None]*inv_freq[None, :]
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    q_even = q[..., 0::2]
    q_odd = q[..., 1::2]
    k_even = k[..., 0::2]
    k_odd = k[..., 1::2]
    q_rot_even = q_even*cos - q_odd*sin
    k_rot_even = k_even*cos - k_odd*sin
    q_rot_odd = q_even * sin + q_odd*cos
    k_rot_odd = k_even * sin + k_odd*cos
    k_rot = torch.stack([k_rot_even, k_rot_odd], dim=-1).flatten(-2)
    q_rot = torch.stack([q_rot_even, q_rot_odd], dim=-1).flatten(-2)
    return q_rot, k_rot

    # 2. Split into even/odd pairs
    # 3. Apply rotation


In [10]:
# 🧪 Debug
q = torch.randn(1, 8, 16)
k = torch.randn(1, 8, 16)
qr, kr = apply_rope(q, k)
print('Shape preserved:', qr.shape == q.shape)
print('Norm preserved:', torch.allclose(q.norm(dim=-1), qr.norm(dim=-1), atol=1e-4))

Shape preserved: True
Norm preserved: True


In [11]:
# ✅ SUBMIT
from torch_judge import check
check('rope')


🧪 Testing: Rotary Position Embedding (RoPE) (Hard)
──────────────────────────────────────────────────
  ✅ [1/4] Output shapes (5.1ms)
  ✅ [2/4] Preserves norm (17.7ms)
  ✅ [3/4] Relative position property (17.1ms)
  ✅ [4/4] Gradient flow (21.1ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (61.0ms total)
  Progress saved. Run status() to see your dashboard.

